In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import pickle

In [ ]:
df_raw = pd.read_csv('salesdaily.csv')
date_col = df_raw.columns[0]
df_raw = df_raw.melt(id_vars=[date_col], var_name='atc_code', value_name='daily_sales')
df_raw['date'] = pd.to_datetime(df_raw[date_col], errors='coerce')
df_raw = df_raw.dropna(subset=['date'])
df_raw['daily_sales'] = pd.to_numeric(df_raw['daily_sales'], errors='coerce').fillna(0)

atc_map = {
    'M01AB': 'Diclofenac', 'M01AE': 'Ibuprofen', 'N02BA': 'Aspirin',
    'N02BE': 'Crocin', 'N05B': 'Alprazolam', 'N05C': 'Sedatives',
    'R03': 'Asthalin', 'R06': 'Cetirizine'
}
df_raw['medicine_name'] = df_raw['atc_code'].map(atc_map)

rows = []
for _, row in df_raw.iterrows():
    base = float(row['daily_sales'])
    past_7 = max(1, int(base * 7 * random.uniform(0.8, 1.2)))
    rows.append({
        'date': row['date'].strftime('%Y-%m-%d'),
        'medicine_name': row['medicine_name'],
        'category': row['atc_code'],
        'past_7_day_sales': past_7,
        'past_30_day_avg_sales': int(base * 30),
        'current_stock': int(past_7 * random.uniform(0.5, 1.5)),
        'days_to_expiry': random.randint(10, 400),
        'price_per_unit': random.randint(15, 350),
        'daily_sales_real': base
    })

store_df = pd.DataFrame(rows)
store_df.to_csv('medstock_store_real.csv', index=False)
print(f'Created {len(store_df)} rows -> medstock_store_real.csv')
store_df.head()

## Feature Engineering → FINAL_TRAINING_DATA_NO_LEAKAGE.csv

In [ ]:
df = pd.read_csv('medstock_store_real.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['medicine_name', 'date'])

df['past_7_day_sales'] = df.groupby('medicine_name')['daily_sales_real'].transform(
    lambda x: x.shift(1).rolling(7, min_periods=1).sum()
)
df['past_30_day_avg_sales'] = df.groupby('medicine_name')['daily_sales_real'].transform(
    lambda x: x.shift(1).rolling(30, min_periods=1).mean()
)
df['next_7_day_demand'] = df.groupby('medicine_name')['daily_sales_real'].transform(
    lambda x: x.shift(-7).rolling(7, min_periods=1).sum()
)
df['month'] = df['date'].dt.month
df = df.dropna(subset=['past_7_day_sales', 'past_30_day_avg_sales', 'next_7_day_demand'])

live = pd.read_csv('live_data.csv').iloc[-1]
df['live_temp']     = live['live_temp']
df['live_humidity'] = live['live_humidity']
df['is_rainy']      = live['is_rainy']
df['fever_trend']   = live['fever_trend']
df['allergy_trend'] = live['allergy_trend']

df.to_csv('FINAL_TRAINING_DATA_NO_LEAKAGE.csv', index=False)
print(f'Clean dataset: {len(df)} rows')
df[['date','medicine_name','past_7_day_sales','next_7_day_demand']].head()

In [ ]:
df = pd.read_csv('FINAL_TRAINING_DATA_NO_LEAKAGE.csv')
print(df.shape)
df.head()

In [ ]:
sns.countplot(data=df, x='medicine_name')
plt.xticks(rotation=30)
plt.tight_layout()

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x='medicine_name', y='past_7_day_sales')
plt.xlabel('Medicine')
plt.ylabel('Past 7-Day Sales')
plt.xticks(rotation=30)
plt.tight_layout()

In [ ]:
df['month'] = pd.to_datetime(df['date']).dt.month
df.groupby('month')['next_7_day_demand'].mean().plot(kind='bar', figsize=(8,5))
plt.xlabel('Month')
plt.ylabel('Avg 7-Day Demand')
plt.title('Average Demand by Month')
plt.tight_layout()

In [ ]:
FEATURE_COLS = [
    'past_7_day_sales', 'past_30_day_avg_sales', 'current_stock',
    'days_to_expiry', 'price_per_unit', 'live_temp', 'live_humidity',
    'is_rainy', 'fever_trend', 'allergy_trend', 'month'
]
TARGET = 'next_7_day_demand'

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.read_csv('FINAL_TRAINING_DATA_NO_LEAKAGE.csv')
df['month'] = pd.to_datetime(df['date']).dt.month
df = df.dropna(subset=FEATURE_COLS + [TARGET])

X = df[FEATURE_COLS]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

model = RandomForestRegressor(
    n_estimators=200, max_depth=20, min_samples_leaf=3,
    n_jobs=-1, random_state=42
)
model.fit(X_train, y_train)
print(f'Train size: {len(X_train)}  Test size: {len(X_test)}')

In [ ]:
pred = model.predict(X_test)
print(f'MAE : {mean_absolute_error(y_test, pred):.2f} units')
print(f'R²  : {r2_score(y_test, pred):.4f}')
print(f'Train R²: {model.score(X_train, y_train):.4f}')

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_test, y=pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual next-7-day demand')
plt.ylabel('Predicted')
plt.title('Actual vs Predicted')
plt.tight_layout()

In [ ]:
feat_imp = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
feat_imp.plot(kind='bar', figsize=(10, 5), title='Feature Importances')
plt.tight_layout()

In [ ]:
pickle.dump(model, open('model_rf.pkl', 'wb'))
print('model_rf.pkl saved')
print('Feature order for API:', FEATURE_COLS)